Import the required libraries and profiling tool.

In [ ]:
import wandb
import pandas as pd
import data_profiling
from data_profiling import ProfileReport

Initialise a W&B run and fetch the latest `sample.csv` artifact. This notebook is versioned in addition to the data.

In [ ]:
run = wandb.init(project='nyc_airbnb', group="eda", save_code=True)  # save_code=True to track the notebook in wandb
local_path = run.use_artifact('sample.csv:latest').file()  # download artifact and return local path
df = pd.read_csv(local_path)

A quick look at the first rows of the dataframe before running the profiling report.

In [ ]:
df.head(3)

The profiling reveals issues: `last_review` is stored as a string instead of a datetime and `price` contains zeros and values above $350 that are implausible for short-term rentals. Missing values in `last_review` and `reviews_per_month` will be handled by the inference pipeline.

In [ ]:
profile = ProfileReport(df, title="Pre Cleanup Profiling Report")
profile.to_notebook_iframe()

Based on stakeholder input, prices outside \$10–\$350 are considered outliers and dropped. `last_review` is converted to datetime so downstream components can treat it correctly.

In [ ]:
df = df[df['price'].between(10, 350)].copy()
df['last_review'] = pd.to_datetime(df['last_review'])

Re-run the profiling to confirm that our fixes where applied

In [ ]:
profile = ProfileReport(df, title="Post Cleanup Profiling Report")
profile.to_notebook_iframe()

Close the W&B run to flush all logged data and mark the EDA step as complete.

In [ ]:
run.finish()